# Train a local literary sentence-transformer model

This notebook demonstrates how to train a local sentence-transformer model from a user-supplied literary corpus.

The original training corpus used in this project is not included because it contains copyrighted text. To run this notebook, provide a local plain-text file with one sentence or short text unit per line.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from literary_nlp.training import (
    load_training_sentences,
    build_adjacent_sentence_pairs,
    train_epoch_grid,
)

In [ ]:
# Training configuration

CORPUS_PATH = Path("data/training_corpus.txt")

OUTPUT_DIR = Path("../models")

BASE_MODEL_NAME = "bert-base-uncased"
MAX_SEQ_LENGTH = 256
BATCH_SIZE = 16
WARMUP_STEPS = 100

EPOCH_VALUES = (1, 2, 4, 8)

PAIRING_STEP = 2  # 2 = non-overlapping adjacent pairs; 1 = overlapping adjacent pairs

In [ ]:
# Load local training corpus

sentences = load_training_sentences(CORPUS_PATH)

print(f"Loaded {len(sentences)} training sentences / text units.")
print("First three entries:")
for sentence in sentences[:3]:
    print("-", sentence[:120])

In [ ]:
# Build adjacent-sentence training pairs

training_examples = build_adjacent_sentence_pairs(
    sentences,
    step=PAIRING_STEP,
)

print(f"Created {len(training_examples)} training pairs.")

In [ ]:
# Train sentence-transformer models

saved_model_paths = train_epoch_grid(
    training_examples=training_examples,
    output_dir=OUTPUT_DIR,
    epoch_values=EPOCH_VALUES,
    base_model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
)

saved_model_paths

## Notes

This notebook trains separate models for each value in `EPOCH_VALUES`. The evaluation notebook can then compare these saved models.

The default pairing strategy uses adjacent text units as positive pairs and `MultipleNegativesRankingLoss`, where other examples in the same batch act as implicit negatives.